# 🧠 Neural Networks Intuition
### Pre-work Notebook 3 of 3

**Time:** ~60 minutes &nbsp;|&nbsp; **Level:** No math background needed &nbsp;|&nbsp; **Requires NumPy**

---

## Why this notebook?

GPT-4, Gemini, Claude — all of them are neural networks.

You don't need to know every detail. But you do need to understand the core loop:

```
1. The model makes a prediction
2. We measure how wrong it is  (loss)
3. We nudge the model slightly in the right direction  (gradient descent)
4. Repeat millions of times  →  the model gets better
```

Everything in M00–M18 builds on this loop. After this notebook, you'll have the mental model to understand it.

---

## Structure of this notebook

| Section | Topic | Time |
|---|---|---|
| 1 | What is a neural network? (the picture) | 10 min |
| 2 | Classification — the model picks from options | 10 min |
| 3 | Weights — the dials the model tunes | 15 min |
| 4 | Loss + Gradient Descent — how learning happens | 15 min |
| 5 | CHALLENGE — Watch a network learn | 10 min (optional) |

In [ ]:
import numpy as np
print("NumPy ready!")

---
## Section 1: What Is a Neural Network?

### The picture you need

**Analogy:** Imagine a factory assembly line with 3 stations:

```
RAW MATERIAL          PROCESSING STATIONS            FINAL PRODUCT
   (Input)           (Hidden Layers)                   (Output)

  [word]  ──→  [Station 1] ──→ [Station 2] ──→  [next word probability]
```

- **Input layer:** The data going in (a word, an image, a sentence)
- **Hidden layers:** Where the transformation happens (the 'thinking' part)
- **Output layer:** The final answer (probabilities for each possible next word)

Each station (layer) is made of **neurons** — simple math operations.
Each connection between neurons has a **weight** — a number that controls how much signal passes through.

In [ ]:
# ── THE NEURAL NETWORK PICTURE IN CODE ───────────────────────────────

# Let's build the SIMPLEST possible neural network:
# - 2 inputs  (two features about a word)
# - 1 output  (a score)

# A single neuron does this:
#   output = (input1 × weight1) + (input2 × weight2) + bias
#   Then squeeze the output through an activation function

def neuron(inputs, weights, bias):
    """A single neuron: weighted sum of inputs + bias."""
    # Multiply each input by its weight and sum everything up
    total = sum(i * w for i, w in zip(inputs, weights)) + bias
    return total

# Imagine a word described by 2 features:
# [frequency_in_text, average_word_length_nearby]
word_features = [0.8, 0.3]

# Weights are just numbers — initially random, improved through training
weights = [0.5, 0.2]
bias = 0.1   # a constant offset — like a baseline score

output = neuron(word_features, weights, bias)
print(f"Input features: {word_features}")
print(f"Weights:        {weights}")
print(f"Bias:           {bias}")
print(f"Neuron output:  {output:.3f}")
print()
print("The output is: 0.8×0.5 + 0.3×0.2 + 0.1 =", 0.8*0.5 + 0.3*0.2 + 0.1)

In [ ]:
# ── STACKING NEURONS INTO LAYERS ─────────────────────────────────────

# A layer is just multiple neurons working in parallel
# Each neuron looks at the same input but with different weights
# → Each neuron learns to detect a different pattern

# With NumPy, a whole layer is just a matrix multiplication
# (You don't need to understand the math — just the idea)

np.random.seed(42)   # makes random numbers reproducible

# Input: a word described by 4 features
word_input = np.array([0.8, 0.3, 0.5, 0.1])

# Layer 1: 3 neurons, each with 4 weights (one per input feature)
# Shape: (3 neurons × 4 inputs)
layer1_weights = np.random.randn(3, 4) * 0.1   # small random numbers
layer1_bias    = np.zeros(3)                     # biases start at 0

# Forward pass through layer 1
layer1_output = np.dot(layer1_weights, word_input) + layer1_bias
print("Layer 1 output (3 values):", layer1_output.round(3))

# Layer 2: 2 neurons, takes layer1's output as input
layer2_weights = np.random.randn(2, 3) * 0.1
layer2_bias    = np.zeros(2)

layer2_output = np.dot(layer2_weights, layer1_output) + layer2_bias
print("Layer 2 output (2 values):", layer2_output.round(3))

print()
print("The data flowed: 4 inputs → 3 neurons → 2 neurons")
print("This is a forward pass — data moving forward through the network.")

---
## Section 2: Classification — The Model Picks From Options

### The most important concept for understanding LLMs

**Classification** = "Given an input, pick the most likely option from a fixed list."

Examples:
- Email classifier: pick from `[spam, not spam]`
- Sentiment: pick from `[positive, negative, neutral]`
- **Next token prediction: pick from `[the, cat, sat, on, ... 50,000 words]`** ← this is LLMs

That last one is exactly what GPT does. Every single word it generates is a classification problem:
> "Given everything I've seen so far, what is the most likely next word?"

The model runs this classification **billions of times** during training (once per word in all of the internet's text) until it gets very good at it.

In [ ]:
# ── CLASSIFICATION IN ACTION ──────────────────────────────────────────

# Tiny example: classify a review as POSITIVE or NEGATIVE
# Input: [exclamation_marks, negative_words, length_score]
# Output: probability of POSITIVE vs NEGATIVE

def softmax(scores):
    """Converts raw scores into probabilities (from Notebook 2)."""
    exp_scores = np.exp(scores - np.max(scores))   # subtract max for numerical stability
    return exp_scores / exp_scores.sum()


# A 1-layer classifier
# Weights learned what features matter for each class:
#                      [exclamations, negative_words, length]
weights_positive = np.array([ 0.8,         -0.9,         0.3])  # exclamations → positive
weights_negative = np.array([-0.3,          0.9,        -0.2])  # negative words → negative
bias = np.array([0.1, -0.1])

def classify_review(features):
    """Classify a review as POSITIVE or NEGATIVE."""
    # Score for each class
    score_positive = np.dot(weights_positive, features) + bias[0]
    score_negative = np.dot(weights_negative, features) + bias[1]
    scores = np.array([score_positive, score_negative])

    # Convert to probabilities
    probs = softmax(scores)
    labels = ['POSITIVE', 'NEGATIVE']
    predicted = labels[np.argmax(probs)]

    return predicted, probs


# Test: "Amazing product!! Love it!"
# features: [exclamations=high, negative_words=none, length=medium]
review_amazing = np.array([0.9, 0.0, 0.5])
pred, probs = classify_review(review_amazing)
print(f"'Amazing product!!' → {pred} (confidence: {max(probs):.0%})")

# Test: "Terrible. Broke after one day."
# features: [exclamations=none, negative_words=high, length=short]
review_bad = np.array([0.0, 0.8, 0.2])
pred, probs = classify_review(review_bad)
print(f"'Terrible. Broke after one day.' → {pred} (confidence: {max(probs):.0%})")

In [ ]:
# ✏️  YOUR TURN — Section 2
# Fill in the ___ blanks

# The model is predicting the next word after "The weather is very ___"
# It's a 3-class classification: [hot, cold, unpredictable]

# The model's raw scores for each candidate word:
candidate_words = ['hot', 'cold', 'unpredictable']
raw_scores = np.array([6.2, 3.8, 1.1])

# Step 1: Convert to probabilities with softmax
probs = ___(raw_scores)

# Step 2: Print each word with its probability
print("Predictions for 'The weather is very ___':")
for word, prob in zip(candidate_words, probs):
    bar = '█' * int(prob * 40)
    print(f"  {word:<15} {prob:.1%}  {bar}")

# Step 3: The model picks the highest probability word
winner_index = np.argmax(___)
winner_word  = candidate_words[___]
print(f"\nModel predicts: '{winner_word}'")

# ✅ 'hot' should win

---
## Section 3: Weights — The Dials the Model Tunes

### What are weights, and how does the model learn them?

**Analogy:** Imagine a mixing board in a recording studio. It has hundreds of dials (volume, bass, treble for each track). A bad mix sounds wrong. A good engineer adjusts the dials until it sounds right.

A neural network has millions of these dials — called **weights**.

Training is the process of automatically adjusting all these dials so the network's output matches the correct answer.

- **Before training:** weights are random → output is garbage
- **During training:** weights are nudged slightly after each mistake
- **After training:** weights encode all the patterns learned from data

GPT-4 has ~1.8 trillion weights. Every single one was adjusted during training.

In [ ]:
# ── WEIGHTS: BEFORE AND AFTER TRAINING ───────────────────────────────

# Simple task: predict if a review is positive (1) or negative (0)
# Input: [has_exclamation, has_word_great, has_word_terrible]
# Output: probability of being positive

def sigmoid(x):
    """Squeezes any number into a range of 0 to 1 (useful for probabilities)."""
    return 1 / (1 + np.exp(-x))

def predict(features, weights, bias):
    """Make a prediction: 1 = positive, 0 = negative."""
    score = np.dot(features, weights) + bias
    probability = sigmoid(score)   # squeeze into 0-1 range
    return probability


# Training examples: [has_exclamation, has_great, has_terrible] → label
X_train = np.array([
    [1, 1, 0],   # "Great product!"   → positive
    [0, 0, 1],   # "Terrible quality" → negative
    [1, 0, 0],   # "Works!"           → positive
    [0, 1, 0],   # "Great value"      → positive
    [0, 0, 1],   # "Terrible smell"   → negative
])
y_train = np.array([1, 0, 1, 1, 0])   # 1 = positive, 0 = negative

# BEFORE training: random weights
np.random.seed(0)
weights_random = np.random.randn(3) * 0.1
bias_random = 0.0

print("=== BEFORE training ===")
print(f"Weights: {weights_random.round(3)}")
for features, label in zip(X_train, y_train):
    pred = predict(features, weights_random, bias_random)
    correct = '✅' if round(pred) == label else '❌'
    print(f"  {correct} Features {features} → predicted {pred:.2f}, actual {label}")

In [ ]:
# ✏️  YOUR TURN — Section 3
# Fill in the ___ blanks

# You are going to manually set weights that you THINK will work well
# Think about it:
#   has_exclamation → slightly positive signal  → weight should be slightly positive
#   has_great       → strongly positive signal  → weight should be a larger positive number
#   has_terrible    → strongly negative signal  → weight should be a negative number

# Set weights to reflect your intuition
# e.g. weights = [exclamation_weight, great_weight, terrible_weight]
my_weights = np.array([___, ___, ___])   # try values between -2.0 and 2.0
my_bias = 0.0

X_train = np.array([[1,1,0],[0,0,1],[1,0,0],[0,1,0],[0,0,1]])
y_train = np.array([1, 0, 1, 1, 0])

print("=== YOUR hand-tuned weights ===")
print(f"Weights: {my_weights}")
correct_count = 0
for features, label in zip(X_train, y_train):
    pred = predict(features, my_weights, my_bias)
    is_correct = round(pred) == label
    if is_correct:
        correct_count += 1
    icon = '✅' if is_correct else '❌'
    print(f"  {icon} Features {features} → predicted {pred:.2f}, actual {label}")

print(f"\nAccuracy: {correct_count}/{len(y_train)} correct")
print("Can you get 5/5?  Hint: 'great' needs a positive weight, 'terrible' needs a negative weight.")

---
## Section 4: Loss and Gradient Descent — How Learning Happens

### The core loop of ALL machine learning

You just tried to manually tune weights. Imagine doing that for 1.8 trillion weights — impossible.

Instead, we let the model tune itself automatically. Here's how:

**Step 1 — Make a prediction** (forward pass)

**Step 2 — Measure the error** using a **loss function**
> Loss = how wrong you are. Loss of 0 = perfect. Higher loss = more wrong.

**Step 3 — Nudge weights in the right direction** using **gradient descent**
> Analogy: You're blindfolded on a hilly landscape. Your goal is to reach the lowest point (minimum loss). You can't see — but you can feel the slope under your feet. Take a small step downhill. Repeat.

**Step 4 — Repeat millions of times**
> Each repeat is called an **iteration** or **training step**.

In [ ]:
# ── LOSS FUNCTION ─────────────────────────────────────────────────────

# The loss function measures how wrong the model is
# We use Mean Squared Error (MSE): average of (prediction - actual)^2

def mse_loss(predictions, actuals):
    """Mean Squared Error — how wrong are the predictions on average?"""
    errors = predictions - actuals         # how wrong for each example
    squared_errors = errors ** 2           # square to make all positive, penalise big errors more
    return np.mean(squared_errors)         # average across all examples


# Random weights → bad predictions → high loss
np.random.seed(0)
bad_weights = np.random.randn(3) * 0.1

X_train = np.array([[1,1,0],[0,0,1],[1,0,0],[0,1,0],[0,0,1]], dtype=float)
y_train = np.array([1, 0, 1, 1, 0], dtype=float)

bad_preds = np.array([predict(x, bad_weights, 0) for x in X_train])
bad_loss  = mse_loss(bad_preds, y_train)
print(f"Random weights loss: {bad_loss:.4f}")

# Good weights → better predictions → lower loss
good_weights = np.array([0.5, 2.0, -2.0])   # exclamation helps, great=positive, terrible=negative
good_preds = np.array([predict(x, good_weights, 0) for x in X_train])
good_loss  = mse_loss(good_preds, y_train)
print(f"Good weights loss:   {good_loss:.4f}")

print()
print(f"Lower loss = better model. Training tries to minimise loss automatically.")

In [ ]:
# ── GRADIENT DESCENT — WATCH THE MODEL LEARN ─────────────────────────

# The model starts with random weights and improves step by step
# learning_rate controls how big each step is

np.random.seed(42)
weights = np.random.randn(3) * 0.1   # start random
bias    = 0.0
learning_rate = 0.5                   # how big each learning step is

print("Starting weights:", weights.round(3))
print()
print(f"{'Step':<8} {'Loss':<12} {'Weights':<40}")
print("-" * 65)

for step in range(20):   # 20 training steps

    # FORWARD PASS: make predictions with current weights
    preds = np.array([predict(x, weights, bias) for x in X_train])

    # COMPUTE LOSS: how wrong are we?
    loss = mse_loss(preds, y_train)

    # BACKWARD PASS: calculate which direction to nudge each weight
    # (simplified gradient — derivative of MSE loss)
    errors = preds - y_train
    grad_weights = np.dot(X_train.T, errors * preds * (1 - preds)) / len(y_train)
    grad_bias    = np.mean(errors * preds * (1 - preds))

    # UPDATE WEIGHTS: take a step in the downhill direction
    weights -= learning_rate * grad_weights   # -= means: subtract and update
    bias    -= learning_rate * grad_bias

    # Print progress every 2 steps
    if step % 2 == 0:
        print(f"{step:<8} {loss:<12.4f} {str(weights.round(3)):<40}")

print()
print("Watch the loss go DOWN and weights settle into useful values!")
print(f"\nFinal weights: {weights.round(3)}")
print("Notice: 'great' weight is positive, 'terrible' weight is negative — model learned this!")

In [ ]:
# ✏️  YOUR TURN — Section 4
# Fill in the ___ blanks

# After training above, let's test the model on NEW reviews it has never seen

# Test examples: [has_exclamation, has_great, has_terrible]
test_cases = [
    ([1, 1, 0], "Great experience!"),         # expected: positive
    ([0, 0, 1], "Terrible, very disappointed"), # expected: negative
    ([1, 0, 0], "Fast delivery!"),              # expected: probably positive
    ([0, 1, 1], "Great idea but terrible price") # expected: mixed — tricky!
]

print("=== Model predictions on new reviews ===")
for features, review_text in test_cases:
    features_array = np.array(___)                     # convert to numpy array
    prob_positive  = predict(___, weights, bias)        # make prediction
    label = 'POSITIVE' if prob_positive > ___ else 'NEGATIVE'   # threshold at 0.5

    print(f"  '{review_text}'")
    print(f"  → {label} ({prob_positive:.1%} positive probability)")
    print()

---
## Section 5: CHALLENGE — Connect Everything

> **Optional — 10 minutes**

**The big picture:**

GPT is a giant neural network trained on trillions of words.

For every word in its training data, it did:
1. Look at all previous words (context)
2. **Predict** the next word (classification over 50,000 options)
3. Measure **loss** (how wrong was the prediction?)
4. **Update** 1.8 trillion weights slightly in the right direction
5. Repeat for the next word. And the next. And the next.

After doing this for months on thousands of GPUs, GPT learned to predict text so well that it appears to understand language.

In this challenge, you simulate that exact process — but on a 3-word vocabulary.

In [ ]:
# 🏆 CHALLENGE — Mini Language Model
#
# Task: Train a tiny model to predict the next word
# Vocabulary: ['cat', 'sat', 'mat'] → mapped to [0, 1, 2]
# Training data: "cat sat" → predict 'mat', "cat" → predict 'sat'
#
# Fill in the ___ blanks

# Vocabulary
vocab = {'cat': 0, 'sat': 1, 'mat': 2}
id_to_word = {v: k for k, v in vocab.items()}   # reverse lookup

def softmax(scores):
    exp_scores = np.exp(scores - np.max(scores))
    return exp_scores / exp_scores.sum()

# Training data: (context_word_id, next_word_id)
# "cat" → "sat", "sat" → "mat"
training_pairs = [(0, 1), (1, 2)]   # (cat→sat), (sat→mat)

# Model: a weight matrix — one row per input word, one column per output word
# Shape: (3 inputs × 3 outputs)
np.random.seed(0)
W = np.random.randn(3, 3) * 0.1   # start with small random weights

print("Initial weights (rows=input, cols=output: cat/sat/mat)")
print(W.round(3))
print()

# Training loop
learning_rate = 0.5

for epoch in range(30):   # 30 passes over training data
    total_loss = 0

    for context_id, target_id in training_pairs:
        # Forward pass: get probabilities for each next word
        scores = W[context_id]              # look up row for current word
        probs  = softmax(___)               # convert to probabilities

        # Loss: negative log probability of the correct word
        # (lower = model was more confident about the right answer)
        loss = -np.log(probs[___] + 1e-10)  # +1e-10 prevents log(0)
        total_loss += ___

        # Backward pass: compute gradient (which direction to update weights)
        grad = probs.copy()
        grad[target_id] -= 1              # correct answer gets negative nudge

        # Update weights for the context word row
        W[context_id] -= learning_rate * grad

    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | Loss: {total_loss:.4f}")

print()

# Test the trained model
print("=== Trained model predictions ===")
for word, word_id in vocab.items():
    probs = softmax(W[word_id])
    predicted_id = np.argmax(probs)
    predicted_word = id_to_word[predicted_id]
    print(f"  After '{word}', model predicts: '{predicted_word}' ({probs[predicted_id]:.1%} confident)")

print()
print("If training worked: 'cat'→'sat' and 'sat'→'mat' should be predicted correctly.")

---
## ✅ Notebook 3 Summary — The Complete Picture

| Concept | What it is | One-line explanation |
|---|---|---|
| **Neuron** | Basic unit | Weighted sum of inputs + activation function |
| **Layer** | Group of neurons | Multiple neurons processing input in parallel |
| **Weights** | Tunable numbers | The 'dials' the model learns to adjust |
| **Forward pass** | Making a prediction | Data flows input → layers → output |
| **Classification** | Picking from options | Every LLM prediction is a classification |
| **Loss** | How wrong you are | 0 = perfect, higher = more wrong |
| **Gradient descent** | Learning algorithm | Nudge weights downhill toward lower loss |
| **Training** | The full loop | Forward → loss → gradient → update → repeat |

---

## 🎉 You're Ready for M00

You now have the mental model to understand everything in the GenAI course:

- **Python** — you can read and write the code that appears in every notebook
- **Vectors** — you understand what word embeddings are
- **Softmax** — you understand how a model picks the next token
- **Neural networks** — you understand what training means and why it works

M00 will go deeper on all of this — but you won't be lost. You'll recognise the concepts and build on them.

---

**Before the session:** Read `cheatsheet.md` — a one-page summary of everything you've learned.

> 💡 **What M00 adds:** You'll see how attention (the core of transformers) is built from dot products between vectors — exactly what you practiced in Notebook 2. Now you have the foundation to understand it.